# Chapter 16 Exercise Solutions

These solutions exercise the trigger, evidence, runtime limits, the checkpoint, and the edit path. They run in mock mode without an external model.

In [1]:
import sys, tempfile
from datetime import date
from pathlib import Path

START = Path.cwd().resolve()
REPO_ROOT = next(
    path for path in (START, *START.parents)
    if (path / "ch16_decision" / "scripts").is_dir()
)
SCRIPT_DIR = REPO_ROOT / "ch16_decision" / "scripts"
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(SCRIPT_DIR))

from build_database import build
from signal_monitor import (
    evaluate_hcp_digital_signal, read_trigger, confirm_signal, default_case_id)
from runtime import AgentRuntime, build_decision_request
from memory import CaseStore
from models import HumanDisposition, OutcomeEvent
from config import LATER_DECISION_DATE

build()  # deterministic synthetic commercial environment
WORK = Path(tempfile.mkdtemp())
print("environment built; working directory ready")

signal = evaluate_hcp_digital_signal(date(2026, 7, 14))
case_id = default_case_id()
request = confirm_signal(signal, build_decision_request(
    'first', case_id=case_id, signal_id=signal.signal_id,
    evidence_date=signal.evidence_date))
def fresh_runtime(name, **kw):
    return AgentRuntime(mock=True, store=CaseStore(WORK / f'{name}.sqlite'),
                        checkpoint_path=WORK / f'{name}_ckpt.sqlite', **kw)


environment built; working directory ready


## Exercise 1: Change the trigger threshold

In [2]:
import signal_monitor
original = signal_monitor.TRIGGER['min_engagement_rise']
signal_monitor.TRIGGER['min_engagement_rise'] = 1.10  # above the observed 103%
print('candidate at 110% threshold:', evaluate_hcp_digital_signal(date(2026, 7, 14)))
signal_monitor.TRIGGER['min_engagement_rise'] = original

candidate at 110% threshold: None


## Exercise 2: Raise claims maturity and NRx growth

In [3]:
import signal_monitor
signal_monitor._claims_maturity = lambda as_of: 0.95
signal_monitor._nrx_growth = lambda: 0.40
print('candidate when evidence is settled:', evaluate_hcp_digital_signal(date(2026, 7, 14)))

candidate when evidence is settled: None


## Exercise 3: Remove the experiment result

In [4]:
from tools import run_tool
first_read = run_tool('get_experiment_evidence', 'first')
print('first-date experiment read:', [e.evidence_id for e in first_read])
causal = [e.evidence_id for e in first_read if e.causal_status == 'causal']
print('causal evidence for scale:', causal or 'none -> avoid scale')

first-date experiment read: ['EXP-NONE']
causal evidence for scale: none -> avoid scale


## Exercise 4: Reverse the experiment interval

In [5]:
from evaluation import load_cases, run_case
reversed_case = next(c for c in load_cases('development')
                     if c.case_id == 'RV-REVERSED-EXPERIMENT')
reversed_evidence = reversed_case.tool_overrides['get_experiment_evidence'][0]
print('injected estimate:', reversed_evidence.estimate)
reversed_result = run_case(reversed_case, 'mock', Path(tempfile.mkdtemp()))
print('decision class:', reversed_result.decision_class)
print('released control violation:', reversed_result.control_violation)

injected estimate: -0.4 NRx per 100 HCPs
decision class: targeted_scale
released control violation: False


## Exercise 5: Add an overlapping access event

In [6]:
access_case = next(c for c in load_cases('development')
                   if c.case_id == 'RV-ACCESS-EVENT')
access_evidence = access_case.tool_overrides['get_market_events'][0]
print('injected market event:', access_evidence.estimate)
access_result = run_case(access_case, 'mock', Path(tempfile.mkdtemp()))
print('decision class:', access_result.decision_class)
print('terminal state:', access_result.terminal_status)

injected market event: 1 overlapping access event
decision class: bounded_experiment
terminal state: approved


## Exercise 6: Exceed the tool or cost limit

In [7]:
from models import RuntimeLimits
rt = fresh_runtime('limit', limits=RuntimeLimits(max_llm_steps=1))
rt.create_case(signal, request)
st = rt.start_run(case_id, mode='mock')
state = rt.get_run(st.run_id)
print('reviewer disposition:', state.review.disposition)
print('interrupts:', [i.kind for i in state.interrupts])

reviewer disposition: escalate
interrupts: ['budget_exhausted']


## Exercise 7: Restart at the human interrupt

In [8]:
rt2 = fresh_runtime('restart')
rt2.create_case(signal, request)
st2 = rt2.start_run(case_id, mode='mock')
again = fresh_runtime('restart')  # same durable files
print('recovered recommendation:', again.get_run(st2.run_id).analyst.selected_option_name)

recovered recommendation: Reversible matched-market test


## Exercise 8: Edit an option and revalidate

In [9]:
from models import DecisionOption
rt3 = fresh_runtime('edit')
rt3.create_case(signal, request)
st3 = rt3.start_run(case_id, mode='mock')
bad = DecisionOption(name='Over ceiling', description='too big',
    budget_moved_usd=1_500_000, audience='all_endocrinologists', geography='all_dmas',
    duration_weeks=13, reversibility='low', is_experiment=False,
    measurement_design='outcome_monitor')
from runtime import InvalidTransition
try:
    rt3.submit_disposition(st3.run_id, HumanDisposition(
        decision='edit', reviewer='Brand lead', reason='bad edit', edited_option=bad))
except InvalidTransition as error:
    print('blocked by validation:', str(error)[:60])

blocked by validation: Edited option fails deterministic validation: The selected o
